# Optimal NIRCam pointing offset

In this notebook, we will search the optimal target position for our NIRCam imaging observations.
We want to search for close companions so what we are looking for is a small detector patch (roughly 70x70 pixels)
where we can place the science target. We still use the FULL detector array to search for wide companions.

We also use primary dithers, so ideally we would want all our primary dither positions to be free of bad pixels.

And finally, we are carrying short wavelength observations in parallel,
so we want to make sure these images are also as free as possible of bad pixels,
and we want to make sure that the target does not fall between two detectors.

## Downloading an observation

We will download a processed NIRCam observation from MAST to access a DQ map on the detector.
We use an observation from the GO 2473 cycle 1 program (P.I. Albert) as they are similar to the ones we will carry.

Let us start by only downloading the long wavelength stage 2 product.

In [ ]:
from pathlib import Path
from astroquery.mast import Observations

x, y = 1282, 825
uri_lw = "mast:JWST/product/jw02473052001_02101_00004_nrcblong_cal.fits"
data_path = Path("./data")
data_path.mkdir(exist_ok=True)
path_lw = data_path / uri_lw.split("/")[-1]

Observations.download_file(uri_lw, local_path=path_lw);

## Looking at the metadata

Let's take a quick look at the science data before we proceed to our pointing optimisation.
Here are all the extensions in a JWST stage 2 image.

In [ ]:
from astropy.io import fits

hdul = fits.open(path_lw)
hdul.info()

We can extract the relevant extensions and headers, and check when the data was processed.

In [ ]:
hdr = hdul[0].header
img = hdul["SCI"].data
dq = hdul["DQ"].data

created_date = hdr["DATE"]
crds_ver = hdr["CRDS_VER"]
crds_ctx = hdr["CRDS_CTX"]
print(f"The file was processed on {created_date} with CRDS version {crds_ver} and context {crds_ctx}")

The file was processed not too long ago so the DQ map should be up to date.

## Looking at the science image

Let's display the science image before we move on to the DQ map.

In [ ]:
from matplotlib import rcParams
import matplotlib.pyplot as plt

rcParams["image.origin"] = "lower"

plt.imshow(img, norm="symlog")
plt.plot(x, y, "r*", label="Target position")
plt.title(path_lw.name)
plt.xlabel("X [pix]")
plt.ylabel("Y [pix]")
plt.colorbar(label="DN/s")
plt.legend()
plt.show()

And let's take a zoomed-in look at our target

In [ ]:
img_size = 70
img_hs = img_size // 2
img_crop = img[y - img_hs: y + img_hs, x - img_hs: x + img_hs]

plt.imshow(img_crop, norm="symlog")
plt.title(f"{path_lw.name} zoomed")
plt.xlabel("X [pix]")
plt.ylabel("Y [pix]")
plt.colorbar(label="DN/s")
plt.show()

## Searching for optimal regions

To avoid messing with the DQ extension, we can just do a nan mask on the images.
(I checked beforehand that this was the same as doing a mask of the `DO_NOT_USE` pixels from the DQ array.)

In [ ]:
import numpy as np
dq_mask = np.isnan(img)

What we want to do next is scan all 70x70 patches in the image and find those with the least bad pixels.

### Scipy implementation

The `find_regions` function is implemented in a separate file to make development easier.


In [ ]:
# TODO: Remove importlib once done writing the code
from importlib import reload
import coldest.pointing
reload(coldest.pointing)
from coldest.pointing import find_regions

from scipy.ndimage import median_filter

In [ ]:
def do_region_search(dq_mask, img, kernel="uniform", forbidden_size=None, show: bool = True):

    region_size = 70
    region_hs = region_size // 2
    n_top = 10


    # Crop the science image and create a filtered version
    img_crop = img[y - region_hs: y + region_hs, x - region_hs: x + region_hs]
    img_crop_mask = img_crop.copy()
    nan_crop_mask = np.isnan(img_crop)
    img_crop_mask[nan_crop_mask] = 0.0
    img_crop_mask[nan_crop_mask] = median_filter(img_crop_mask, size=5)[nan_crop_mask]
    fig = plt.figure()
    plt.imshow(img_crop_mask, norm="symlog")
    plt.title("Median-filtered Real PSF")
    if show:
        plt.show()
    else:
        plt.close(fig)

    if kernel == "weighted":
        kernel = img_crop_mask
        

    # Find the n_top best regions
    best_x, best_y, weighted_mask = find_regions(
        dq_mask,
        region_size,
        kernel=kernel,
        forbidden_size=forbidden_size,
        n_top=n_top,
        return_weighted=True
    )

    # Plot the full frame DQ, weighted DQ and SCI frames with the best regions
    fig, axs = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)
    axs[0].imshow(dq_mask)
    axs[1].imshow(weighted_mask, norm="symlog")
    axs[2].imshow(img, norm="symlog")
    axs[0].set_title("Bad pixel mask")
    axs[1].set_title("Weighted bad pixel mask")
    axs[2].set_title("Science image")
    fig.suptitle("Regions shown on the full detector frame")
    for i in range(n_top):
        axs[0].scatter(best_x[i], best_y[i], marker=f"${i+1}$", color="r")
        axs[1].scatter(best_x[i], best_y[i], marker=f"${i+1}$", color="r")
        axs[2].scatter(best_x[i], best_y[i], marker=f"${i+1}$", color="r")
    if show:
        plt.show()
    else:
        plt.close(fig)

    fig, axs = plt.subplots(2, n_top, figsize=(20, 5), sharex=True, sharey=True)
    for i in range(n_top):
        region_y, region_x = best_y[i], best_x[i]
        region = img[region_y - region_hs: region_y + region_hs, region_x - region_hs: region_x + region_hs]
        nan_count = np.sum(np.isnan(region))
        axs[0, i].imshow(region, norm="symlog")
        axs[0, i].set_title(f"Region {i+1}: {nan_count} DQ")

        img_with_bad = img_crop_mask.copy()
        region_mask = np.isnan(region)
        img_with_bad[region_mask] = np.nan
        axs[1, i].imshow(img_with_bad, norm="symlog")
    fig.suptitle("Regions shown on the science image and bad pixels overlaid on a PSF")
    if show:
        plt.show()
    else:
        plt.close(fig)

### Using a uniform kernel

The simplest thing we can do is define a region size and search for regions that minimize the number of bad pixels uniformly.
We defined a helper function to find the 10 best 70x70 regions.
Let us run it and see the results.

In [ ]:
do_region_search(dq_mask, img, show=True)

First, the function crops the science image to the selected region size and applies a median filter to remove bad pixels.
This is shown on the first plot. It is useful for plotting purposes and for weighted kernels discussed below.

Internally, the function computes the central pixel of the optimal regions before showing them on the DQ map.
We also display the weighted map showing the sum of bad pixels in the subarray centered on each pixel.
And we display the science data to show any cosmetics that may be visible on the detector.
This is all shown on the second plot.

We then take a closer look at the regions by zooming on them in the science image, as shown in the first row of the third plot.
The second row shows where the bad pixels would fall on a PSF in each region.

One thing we may notice here is that some regions have bad pixels fairly close to the core of the PSF.
We have two built-in ways to address this:

1. A "forbidden region" at the center of the PSF where bad pixels are strictly forbidden.
2. A weighted filter where bad pixels in the center have a stronger impact on the "score" than bad pixels on the edges

### Forbidden region

First, let us re-do the search with a forbidden region of 10 pixels at the core.

In [ ]:
do_region_search(dq_mask, img, show=True, forbidden_size=10)

Forbidden pixels are set to infinity, hence the weird looking central image in the second plot.

As we can see on the third figure, some regions from the previous section have been discarded and replace by new ones.

### Weighted kernel

Another thing we can do to improve the search is to use a weighted kernel.
We then use the cleaned PSF image from the first figure as a kernel for the region search.

In [ ]:
do_region_search(dq_mask, img, kernel="weighted", show=True)

This one takes a bit longer due to the convolution used under the hood.

As we can see, the bad pixels near the core are not strictly forbidden, but regions with many bad pixels near the core are disfavored.

### Weighted kernel and forbidden region

We can also combine the improvements from the last two sections to get our final map.

In [ ]:
do_region_search(dq_mask, img, kernel="weighted", forbidden_size=10, show=True)

This will now allow regions with a few more bad pixels, but strongly disfavor bad pixels near the core.

## Dither positions
